In [1]:
import math
import datetime
from datetime import timedelta
from pathlib import Path

import numpy as np
import xarray as xr
import pandas as pd
import parcels
from parcels import StatusCode

In [2]:
reference_time_str = "2025-01-01"

lon_min = -25.15
lon_max = -24.85
lat_min = 16.70
lat_max = 17.00

grid_points_lon = 60
grid_points_lat = 60

refine_lon_fac = 5
refine_lat_fac = 5

integration_days = 10
integration_direction = 1
dt_minutes = 5
v_repel = 0.02

ssc_path = "../data/ssc.nc"
output_dir = "../data"

import json as _json
import os as _os

_params_path = _os.environ.get("SV_RELEASE_PARAMS_JSON")
if _params_path:
    with open(_params_path) as _f:
        _overrides = _json.load(_f)
    globals().update(_overrides)
    print(f"Loaded params from {_params_path}: reference_time_str={reference_time_str}")

In [3]:
reference_time = pd.Timestamp(reference_time_str)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

release_zarr_path = output_dir / f"SV_heatmap_2025_{reference_time.strftime('%Y%m%d')}.zarr"
print(f"Release {reference_time_str} -> {release_zarr_path}")

Release 2025-01-01 -> ../data/SV_heatmap_2025_20250101.zarr


In [4]:
flowfield_env = xr.open_dataset(ssc_path)
flowfield_env_orig = flowfield_env.copy(deep=True)

/home/b/b384140/.conda/envs/parcels/lib/python3.14/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [5]:
flowfield_env.uo.data[:, :, 0] = 0
flowfield_env.uo.data[:, :, -1] = 0
flowfield_env.uo.data[:, 0, :] = 0
flowfield_env.uo.data[:, -1, :] = 0
flowfield_env.vo.data[:, :, 0] = 0
flowfield_env.vo.data[:, :, -1] = 0
flowfield_env.vo.data[:, 0, :] = 0
flowfield_env.vo.data[:, -1, :] = 0

In [6]:
ocean_mask = flowfield_env_orig.uo.mean("time").notnull().astype(float)

In [7]:
mask = ocean_mask.values
nj, ni = mask.shape
u_rep_arr = np.zeros((nj, ni))
v_rep_arr = np.zeros((nj, ni))

for j in range(1, nj - 1):
    for i in range(1, ni - 1):
        if mask[j, i] == 0:
            continue
        if mask[j,   i-1] == 0: u_rep_arr[j, i] += v_repel
        if mask[j,   i+1] == 0: u_rep_arr[j, i] -= v_repel
        if mask[j-1, i  ] == 0: v_rep_arr[j, i] += v_repel
        if mask[j+1, i  ] == 0: v_rep_arr[j, i] -= v_repel

In [8]:
u_repel = xr.DataArray(u_rep_arr, coords=ocean_mask.coords, dims=ocean_mask.dims)
v_repel_field = xr.DataArray(v_rep_arr, coords=ocean_mask.coords, dims=ocean_mask.dims)

flowfield_env["uo"] = flowfield_env.uo + u_repel
flowfield_env["vo"] = flowfield_env.vo + v_repel_field


In [9]:
fieldset = parcels.FieldSet.from_xarray_dataset(
    flowfield_env,
    variables={"U": "uo", "V": "vo"},
    dimensions={"lon": "longitude", "lat": "latitude", "time": "time"},
)

In [10]:
particle_lon = xr.DataArray(
    data=np.linspace(
        lon_min,
        lon_max,
        refine_lon_fac * flowfield_env.sizes["longitude"] - 1,
    ),
    name="plon",
    dims="plon",
)
particle_lat = xr.DataArray(
    data=np.linspace(
        lat_min,
        lat_max,
        refine_lat_fac * flowfield_env.sizes["latitude"] - 1,
    ),
    name="plat",
    dims="plat",
)
particle_lon, particle_lat = xr.broadcast(particle_lon, particle_lat)

n_particles = particle_lon.size
particle_ids = np.arange(n_particles)
print(f"{n_particles} particles")

230656 particles


In [11]:
class SVParticle(parcels.JITParticle):
    # Custom field so we can track each particle's original grid index across releases.
    pid_orig = parcels.Variable("pid_orig", dtype=np.int32)


def CheckOutOfBounds(particle, fieldset, time):
    if particle.state == StatusCode.ErrorOutOfBounds:
        particle.delete()

def CheckError(particle, fieldset, time):
    if particle.state >= 50:
        particle.delete()

In [12]:
pset = parcels.ParticleSet.from_list(
    fieldset=fieldset,
    pclass=SVParticle,
    lon=particle_lon.stack(pid=["plon", "plat"]).load().data,
    lat=particle_lat.stack(pid=["plon", "plat"]).load().data,
    time=np.datetime64(reference_time),
    pid_orig=particle_ids,
)

output_file = pset.ParticleFile(
    name=str(release_zarr_path),
    outputdt=timedelta(hours=1),
)

kernels = (
    pset.Kernel(parcels.AdvectionRK4)
    + pset.Kernel(CheckOutOfBounds)
    + pset.Kernel(CheckError)
)

pset.execute(
    kernels,
    runtime=timedelta(days=integration_days),
    dt=timedelta(minutes=int(integration_direction * dt_minutes)),
    output_file=output_file,
)

print(f"Wrote {release_zarr_path}")

INFO: Output files are stored in ../data/SV_heatmap_2025_20250101.zarr.
  0%|          | 0/864000.0 [00:00<?, ?it/s]

/home/b/b384140/.conda/envs/parcels/lib/python3.14/site-packages/parcels/particledata.py:358: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  np.less_equal(time - np.abs(pd["dt"] / 2), pd["time"], where=np.isfinite(pd["time"]))
/home/b/b384140/.conda/envs/parcels/lib/python3.14/site-packages/parcels/particledata.py:359: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  & np.greater_equal(time + np.abs(pd["dt"] / 2), pd["time"], where=np.isfinite(pd["time"]))
/home/b/b384140/.conda/envs/parcels/lib/python3.14/site-packages/parcels/particledata.py:360: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  | ((np.isnan(pd["dt"])) & np.equal(time, pd["time"], where=np.isfinite(pd["time"])))


100%|██████████| 864000.0/864000.0 [09:30<00:00, 1514.65it/s]
Wrote ../data/SV_heatmap_2025_20250101.zarr
